# DAgger vs RLPD — multi-run analysis

This notebook discovers the result directories produced by `uv run stk-experiment`, checks their artifacts, and compares learning curves and final performance across runs. Seed-level summaries are used for uncertainty estimates; episode-level plots are diagnostic only.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the project root or notebooks directory")

RESULTS_ROOT = PROJECT_ROOT / "results" / "stk"
RESULTS_ROOT

## Discover and validate runs

A complete run must contain metadata, both learning histories, and both final evaluations. Incomplete directories are listed rather than silently discarded.

In [ ]:
REQUIRED_FILES = [
    "metadata.json",
    "dagger_history.npz",
    "rlpd_learning_curve.pt",
    "dagger_final_evaluation.json",
    "rlpd_final_evaluation.json",
]

run_records = []
for metadata_path in sorted(RESULTS_ROOT.glob("seed_*/metadata.json")):
    run_dir = metadata_path.parent
    metadata = json.loads(metadata_path.read_text())
    missing = [name for name in REQUIRED_FILES if not (run_dir / name).exists()]
    run_records.append(
        {
            "run_id": run_dir.name,
            "run_dir": run_dir,
            "seed": int(metadata["seed"]),
            "track": metadata["track"],
            "offline_transitions": int(metadata["offline_transitions"]),
            "complete": not missing,
            "missing": ", ".join(missing),
            "metadata": metadata,
        }
    )

runs = pd.DataFrame(run_records)
if runs.empty:
    raise FileNotFoundError(f"No experiment runs found below {RESULTS_ROOT}")
display(runs.drop(columns=["run_dir", "metadata"]))

complete_runs = runs[runs["complete"]].copy()
if complete_runs.empty:
    raise RuntimeError("No complete runs are available for analysis")

duplicate_seeds = complete_runs[complete_runs.duplicated("seed", keep=False)]
if not duplicate_seeds.empty:
    print(
        "Warning: repeated training seeds are separate reruns, not independent seeds:"
    )
    display(duplicate_seeds[["run_id", "seed"]])

print(
    f"Using {len(complete_runs)} complete runs and {complete_runs.seed.nunique()} unique seeds"
)

In [ ]:
def mean_ci(frame, group_columns, value_column):
    summary = (
        frame.groupby(group_columns, as_index=False)[value_column]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    summary["ci95"] = 1.96 * summary["std"].fillna(0.0) / np.sqrt(summary["count"])
    return summary


def add_mean_curve(fig, summary, x, name, color):
    fig.add_trace(
        go.Scatter(
            x=summary[x],
            y=summary["mean"],
            error_y={"type": "data", "array": summary["ci95"], "visible": True},
            mode="lines+markers",
            name=f"{name} mean ± 95% CI",
            line={"color": color, "width": 4},
        )
    )

## Learning curves

DAgger is indexed by expert-labelled interactions. RLPD is indexed by online environment interactions. Thin lines show individual runs; the thick line and error bars show the mean and normal-approximation 95% confidence interval across runs.

In [ ]:
dagger_rows = []
rlpd_rows = []
for run in complete_runs.itertuples():
    with np.load(run.run_dir / "dagger_history.npz") as history:
        for iteration, (labels, score, loss) in enumerate(
            zip(
                history["n_labels"], history["validation_score"], history["train_loss"]
            ),
            start=1,
        ):
            dagger_rows.append(
                {
                    "run_id": run.run_id,
                    "seed": run.seed,
                    "iteration": iteration,
                    "interactions": int(labels),
                    "return": float(score),
                    "train_loss": float(loss),
                }
            )

    evaluations = torch.load(
        run.run_dir / "rlpd_learning_curve.pt",
        map_location="cpu",
        weights_only=False,
    )
    eval_frequency = int(run.metadata["rlpd_config"]["eval_freq"])
    for checkpoint, score in enumerate(evaluations):
        rlpd_rows.append(
            {
                "run_id": run.run_id,
                "seed": run.seed,
                "checkpoint": checkpoint,
                "interactions": checkpoint * eval_frequency,
                "return": float(score),
            }
        )

dagger_curve = pd.DataFrame(dagger_rows)
rlpd_curve = pd.DataFrame(rlpd_rows)
display(dagger_curve.head(), rlpd_curve.head())

In [ ]:
dagger_summary = mean_ci(dagger_curve, ["iteration"], "return")
mean_labels = dagger_curve.groupby("iteration")["interactions"].mean()
dagger_summary["interactions"] = dagger_summary["iteration"].map(mean_labels)

fig = px.line(
    dagger_curve,
    x="interactions",
    y="return",
    color="run_id",
    markers=True,
    title="DAgger validation return vs expert-labelled interactions",
)
fig.update_traces(opacity=0.25, line={"width": 1})
add_mean_curve(fig, dagger_summary, "interactions", "DAgger", "#1f77b4")
fig.update_layout(
    xaxis_title="Expert-labelled environment interactions", yaxis_title="Mean return"
)
fig.show()

In [ ]:
rlpd_summary = mean_ci(rlpd_curve, ["interactions"], "return")

fig = px.line(
    rlpd_curve,
    x="interactions",
    y="return",
    color="run_id",
    title="RLPD evaluation return vs online interactions",
)
fig.update_traces(opacity=0.25, line={"width": 1})
add_mean_curve(fig, rlpd_summary, "interactions", "RLPD", "#d62728")
fig.update_layout(
    xaxis_title="Online environment interactions", yaxis_title="Mean return"
)
fig.show()

In [ ]:
fig = go.Figure()
add_mean_curve(fig, dagger_summary, "interactions", "DAgger", "#1f77b4")
add_mean_curve(fig, rlpd_summary, "interactions", "RLPD", "#d62728")
fig.update_layout(
    title="Mean learning curves by online environment interactions",
    xaxis_title="Online environment interactions",
    yaxis_title="Mean return",
)
fig.show()

## Final paired evaluation

The final evaluations use the same evaluation seeds for both algorithms. Each training run—not each episode—is the independent experimental unit.

In [ ]:
summary_rows = []
episode_rows = []
for run in complete_runs.itertuples():
    for algorithm, filename in [
        ("DAgger", "dagger_final_evaluation.json"),
        ("RLPD", "rlpd_final_evaluation.json"),
    ]:
        result = json.loads((run.run_dir / filename).read_text())
        summary_rows.append(
            {
                "run_id": run.run_id,
                "seed": run.seed,
                "algorithm": algorithm,
                **result["summary"],
            }
        )
        episode_rows.extend(
            {
                "run_id": run.run_id,
                "training_seed": run.seed,
                "algorithm": algorithm,
                **episode,
            }
            for episode in result["episodes"]
        )

final_summary = pd.DataFrame(summary_rows)
final_episodes = pd.DataFrame(episode_rows)
display(final_summary.sort_values(["seed", "algorithm"]))

In [ ]:
metric_labels = {
    "mean_return": "Mean return",
    "completion_rate": "Completion rate",
    "mean_distance": "Mean distance",
}

for metric, label in metric_labels.items():
    metric_summary = mean_ci(final_summary, ["algorithm"], metric)
    fig = px.bar(
        metric_summary,
        x="algorithm",
        y="mean",
        error_y="ci95",
        color="algorithm",
        title=f"Final {label.lower()} across training runs",
    )
    fig.add_trace(
        go.Scatter(
            x=final_summary["algorithm"],
            y=final_summary[metric],
            mode="markers",
            marker={"color": "black", "size": 8},
            name="Individual runs",
        )
    )
    fig.update_layout(yaxis_title=label, showlegend=False)
    fig.show()

In [ ]:
paired_return = final_summary.pivot(
    index=["run_id", "seed"], columns="algorithm", values="mean_return"
).reset_index()
paired_return["RLPD minus DAgger"] = paired_return["RLPD"] - paired_return["DAgger"]
display(paired_return)

fig = px.bar(
    paired_return,
    x="run_id",
    y="RLPD minus DAgger",
    color="RLPD minus DAgger",
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title="Paired final-return difference for each training run",
)
fig.add_hline(y=0, line_color="black")
fig.update_layout(xaxis_title="Run", yaxis_title="RLPD return − DAgger return")
fig.show()

In [ ]:
fig = px.box(
    final_episodes,
    x="algorithm",
    y="return",
    color="algorithm",
    points="outliers",
    title="Episode-return distributions (descriptive, not independent samples)",
)
fig.show()

episode_by_seed = final_episodes.pivot_table(
    index=["run_id", "training_seed", "seed"],
    columns="algorithm",
    values="return",
).reset_index()
episode_by_seed["RLPD minus DAgger"] = (
    episode_by_seed["RLPD"] - episode_by_seed["DAgger"]
)
display(episode_by_seed.head())

## Interpretation checklist

- Treat training runs/seeds as the independent units, not the individual evaluation episodes.
- Confidence intervals are descriptive when only a few seeds are available; aim for at least 5 and preferably 10 unique seeds.
- DAgger's interaction count is also its expert-query count. RLPD additionally receives the fixed offline dataset shown in `offline_transitions`.
- The combined curve compares online interactions, but it does not imply equal expert-data or compute budgets.
- Check completion rate and distance alongside return, particularly when few policies finish the race.
- Do not choose hyperparameters using the final evaluation results.